# v2 Annotation Stats

Summary statistics over `data/v2_annotations/`.

In [1]:
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

# The notebook lives in tutormoments_build/v2/, so find the repo root by walking
# up to the directory holding AGENTS.md and import the build package from there.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists()
)
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data" / "v2_annotations"
TRANSCRIPTS = DATA_DIR / "source" / "tutoring_provider_a_v2_transcripts.jsonl"
ANNOTATIONS = DATA_DIR / "source" / "tutoring_provider_a_annotations.jsonl"
SPLITS = REPO_ROOT / "tutormoments_build" / "v2" / "splits.json"

print(f"repo root: .../{REPO_ROOT.name}")

repo root: .../tutormoments


## source/tutoring_provider_a_v2_transcripts.jsonl

In [2]:
transcripts = []
with open(TRANSCRIPTS) as f:
    for line in f:
        line = line.strip()
        if line:
            transcripts.append(json.loads(line))

print(f"Total tutoring sessions: {len(transcripts):,}")

Total tutoring sessions: 462


In [3]:
# Rows per session. A v2 transcript row is either a dialogue turn or a screen
# activity row (type "[SCREEN INTERACTION]", "[PAUSE]", ...); only dialogue rows
# carry a turn number, so both counts are reported.
row_counts = [len(r["turns"]) for r in transcripts]
dialogue_counts = [
    sum(1 for t in r["turns"] if t.get("type") == "dialogue") for r in transcripts
]

avg_rows = sum(row_counts) / len(row_counts)
avg_dialogue = sum(dialogue_counts) / len(dialogue_counts)
print(f"Rows per session          — avg: {avg_rows:.0f}, min: {min(row_counts)}, max: {max(row_counts)}")
print(f"Dialogue turns per session — avg: {avg_dialogue:.0f}, min: {min(dialogue_counts)}, max: {max(dialogue_counts)}")

row_types = Counter(t.get("type") for r in transcripts for t in r["turns"])
print("\nRows by type:")
for rtype, n in row_types.most_common():
    print(f"  {rtype}: {n:,}")

Rows per session          — avg: 457, min: 43, max: 2135
Dialogue turns per session — avg: 384, min: 21, max: 1500

Rows by type:
  dialogue: 177,448
  [SCREEN INTERACTION]: 14,309
  [PAUSE]: 8,151
  [SCREEN UPDATE]: 6,235
  [PROBLEM CHANGE]: 4,822
  [OTHER]: 31


In [4]:
# Whitespace-tokenized token length per dialogue turn
all_turn_lengths = []
for r in transcripts:
    for t in r["turns"]:
        if t.get("type") != "dialogue":
            continue
        text = t.get("text", "") or ""
        all_turn_lengths.append(len(text.split()))

avg_tok = sum(all_turn_lengths) / len(all_turn_lengths)
print(f"Whitespace-token length per dialogue turn — avg: {avg_tok:.1f}, min: {min(all_turn_lengths)}, max: {max(all_turn_lengths)}")
print(f"(across {len(all_turn_lengths):,} dialogue turns total)")

Whitespace-token length per dialogue turn — avg: 12.2, min: 1, max: 1070
(across 177,448 dialogue turns total)


In [5]:
# Session duration distribution (in minutes).
# v2 has no start_seconds/end_seconds; each row carries a "timestamp" string,
# either "MM:SS-MM:SS" / "H:MM:SS-H:MM:SS" for a span or a single "MM:SS".
def _seconds(clock):
    total = 0
    for part in clock.split(":"):
        total = total * 60 + int(part)
    return total


def _span(timestamp):
    """(start, end) in seconds, or None where the row has no timestamp."""
    if not timestamp:
        return None
    if "-" in timestamp:
        start, end = timestamp.split("-", 1)
        return _seconds(start), _seconds(end)
    point = _seconds(timestamp)
    return point, point


durations = []
for r in transcripts:
    spans = [s for s in (_span(t.get("timestamp")) for t in r["turns"]) if s]
    if spans:
        durations.append((spans[-1][1] - spans[0][0]) / 60)

avg_dur = sum(durations) / len(durations)
print(f"Session duration (minutes) — avg: {avg_dur:.1f}, min: {min(durations):.1f}, max: {max(durations):.1f}")
print(f"(across {len(durations):,} sessions with timestamps)")

Session duration (minutes) — avg: 54.5, min: 19.9, max: 119.6
(across 462 sessions with timestamps)


## source/tutoring_provider_a_annotations.jsonl

In [6]:
# One row per moment, plus rows for transcripts an annotator reviewed and found
# nothing in ({"transcript_id", "no_key_moments_record"}).
#
# Retracted moments are dropped here, so nothing below counts them. The tool
# retracts a selection the selector withdrew before it reached a second pass:
# they carry one selector annotation and nothing else, and they are the only
# moments in the export with a single annotator. The annotation viewer drops
# them for the same reason (annotation_viewer/records.py) -- a withdrawn
# judgment is not a judgment, and leaving them in would put one next to live
# ones.
RETRACTED = "retracted"

rows = []
with open(ANNOTATIONS) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

all_moment_rows = [r for r in rows if "moment" in r]
moment_rows = [
    r for r in all_moment_rows if r["moment"].get("status") != RETRACTED
]
retracted_rows = [r for r in all_moment_rows if r["moment"].get("status") == RETRACTED]
no_key_rows = [r for r in rows if "no_key_moments_record" in r]

print(f"Total records: {len(rows):,}")
print(f"  moment records:        {len(all_moment_rows):,}")
print(f"    retracted, dropped:  {len(retracted_rows):,}")
print(f"    kept below:          {len(moment_rows):,}")
print(f"  no-key-moment records: {len(no_key_rows):,}")
print(f"Transcripts covered: {len({r['transcript_id'] for r in rows}):,}")


Total records: 1,264
  moment records:        1,214
    retracted, dropped:  3
    kept below:          1,211
  no-key-moment records: 50
Transcripts covered: 334


In [7]:
# Reading annotations
#
# Not every annotation answers the label questions. An annotator who threw the
# moment out answered nothing: a reannotator leaves situation/action/result
# null, an adjudicator leaves payload["final"] null. Everything below reads
# annotations through answered_payload, so a null section is never read as "no".
#
# A few moments carry the same annotator twice in the same role, saved seconds
# apart under successive `revision` numbers -- a re-save, not a second opinion --
# so latest_annotations keeps only the highest revision per (annotator, role)
# and drops project staff, as the ground-truth build does.
from tutormoments_build.v2.build_ground_truth import is_excluded


def latest_annotations(record, role=None):
    """One annotation per (annotator, role): staff dropped, re-saves collapsed."""
    latest = {}
    for a in record["annotations"]:
        if is_excluded(a) or (role is not None and a.get("role") != role):
            continue
        key = (a["annotator_id"], a.get("role"))
        if key not in latest or a.get("revision", 0) > latest[key].get("revision", 0):
            latest[key] = a
    return list(latest.values())


def answered_payload(annotation):
    """The payload section an annotator actually filled in, or None."""
    payload = annotation["payload"]
    if (payload.get("meta") or {}).get("throw_out"):
        return None
    if annotation.get("role") == "adjudicator":
        # An adjudicator answers the same questions inside payload["final"].
        payload = payload.get("final")
        if not payload:
            return None
    # Either section left null is an abstention. Testing both would let a
    # half-filled payload through, and payload_labels would then read the null
    # section as all-False -- an annotator who answered nothing scored as "no".
    if payload.get("situation") is None or payload.get("action") is None:
        return None
    return payload


def thrown_out_by(record, role):
    return any(
        (a["payload"].get("meta") or {}).get("throw_out")
        for a in latest_annotations(record, role)
    )


def cut_point_redrawn(record):
    """Whether a later pass moved the cut point the selector drew.

    The cut point is where the moment is read from, so moving it means the two
    passes did not judge the same excerpt. Any role can record the redraw; in
    this export every one of them is a reannotator's.
    """
    return any(
        (a["payload"].get("meta") or {}).get("redrew_cut_point")
        for a in latest_annotations(record)
    )


def reannotation(record):
    """The second pass on a moment, or None where it never got one."""
    passes = latest_annotations(record, "reannotator")
    return passes[0] if passes else None


role_counts = Counter(a.get("role") for r in moment_rows for a in latest_annotations(r))
print("Annotations per role (re-saves collapsed):")
for role in sorted(role_counts):
    print(f"  {role}: {role_counts[role]:,}")

# The moment record's `status` field is stale and should not be read as either
# a stage or a verdict. It undercounts both: 24 moments say "adjudicated" while
# 72 actually carry an adjudication, and 1 says "thrown_out" while 183 were
# actually thrown out. The tool set it on some transitions and not others.
#
# Both facts are derivable from the annotations themselves, so those are printed
# first and `status` is shown underneath only to document the discrepancy.
n_second = sum(1 for r in moment_rows if latest_annotations(r, "reannotator"))
n_adjudicated = sum(1 for r in moment_rows if latest_annotations(r, "adjudicator"))
n_thrown_out = sum(
    1
    for r in moment_rows
    if thrown_out_by(r, "reannotator") or thrown_out_by(r, "adjudicator")
)

print(f"\nMoments ({len(moment_rows):,}), from the annotations on them:")
print(f"  reached a second pass   {n_second:>6,}")
print(f"  reached an adjudicator  {n_adjudicated:>6,}")
print(f"  thrown out              {n_thrown_out:>6,}  by a second pass or an adjudicator")
print(f"  kept                    {len(moment_rows) - n_thrown_out:>6,}")


Annotations per role (re-saves collapsed):
  adjudicator: 97
  reannotator: 1,211
  selector: 1,211

Moments (1,211), from the annotations on them:
  reached a second pass    1,211
  reached an adjudicator      72
  thrown out                 183  by a second pass or an adjudicator
  kept                     1,028


In [8]:
# How far each moment got through the pipeline. A moment is selected (first
# pass), reannotated (second pass), and a minority then go to one or two
# adjudicators. Either of the later roles can throw a moment out; the selector
# has no throw-out question, since selecting the moment *is* their vote for it.
# Retracted moments are already out (see the load cell), so a moment with no
# second pass here is one still waiting for review rather than a withdrawn one.
n_first = sum(1 for r in moment_rows if latest_annotations(r, "selector"))
n_second = sum(1 for r in moment_rows if reannotation(r))
one_pass_only = sum(
    1 for r in moment_rows if latest_annotations(r, "selector") and not reannotation(r)
)
adjudicator_counts = Counter(len(latest_annotations(r, "adjudicator")) for r in moment_rows)

tossed_second = sum(1 for r in moment_rows if thrown_out_by(r, "reannotator"))
tossed_adjudicator = sum(1 for r in moment_rows if thrown_out_by(r, "adjudicator"))
tossed_both = sum(
    1
    for r in moment_rows
    if thrown_out_by(r, "reannotator") and thrown_out_by(r, "adjudicator")
)
tossed_either = sum(
    1
    for r in moment_rows
    if thrown_out_by(r, "reannotator") or thrown_out_by(r, "adjudicator")
)
# An adjudicator can also reinstate a moment the second pass threw out.
reverted = sum(
    1
    for r in moment_rows
    if any(a["payload"].get("reverted_throw_out") for a in latest_annotations(r, "adjudicator"))
)

print(f"Moment records: {len(moment_rows):,}\n")
print("Passes:")
print(f"  first pass (selected):            {n_first:,}")
print(f"  two passes (selected + second):   {n_second:,}")
print(f"  first pass only, awaiting review: {one_pass_only}")
print(f"  adjudicated once:                 {adjudicator_counts[1]}")
print(f"  adjudicated twice:                {adjudicator_counts[2]}")

print("\nThrown out:")
print(f"  by the second pass:               {tossed_second}")
print(f"  by an adjudicator:                {tossed_adjudicator}")
print(f"  by both:                          {tossed_both}")
print(f"  by either (dropped from the ground truth): {tossed_either}")
print(f"  reinstated by an adjudicator (reverted_throw_out): {reverted}")


Moment records: 1,211

Passes:
  first pass (selected):            1,211
  two passes (selected + second):   1,211
  first pass only, awaiting review: 0
  adjudicated once:                 47
  adjudicated twice:                25

Thrown out:
  by the second pass:               176
  by an adjudicator:                10
  by both:                          3
  by either (dropped from the ground truth): 183
  reinstated by an adjudicator (reverted_throw_out): 2


In [9]:
# What the later passes moved. The stored `moment` record keeps the boundaries
# and cut point the selector drew even after someone moved them: a second pass
# records its redraw in payload["meta"] (`new_start_turn` / `new_end_turn` /
# `new_cut_turn`, null for whatever it left alone), an adjudicator in
# payload["final_*_turn"]. The self-reported flags and the values are counted
# separately, since a handful of flags are set with no value behind them.
#
# A redraw is partial as often as not -- the start and the end move
# independently -- so each edge is counted on its own as well as together.
from tutormoments_build.v2.build_ground_truth import effective_boundaries

second_passes = [(r, reannotation(r)) for r in moment_rows if reannotation(r)]
cut_flagged = cut_moved = bounds_flagged = 0
start_moved = end_moved = both_moved = bounds_moved = 0
for r, a in second_passes:
    meta = a["payload"].get("meta") or {}
    m = r["moment"]
    cut_flagged += bool(meta.get("redrew_cut_point"))
    cut_moved += meta.get("new_cut_turn") not in (None, m["cut_turn"])
    bounds_flagged += bool(meta.get("changed_boundaries"))

    moved_start = meta.get("new_start_turn") not in (None, m["start_turn"])
    moved_end = meta.get("new_end_turn") not in (None, m["end_turn"])
    start_moved += moved_start
    end_moved += moved_end
    both_moved += moved_start and moved_end
    bounds_moved += moved_start or moved_end

print(f"Second passes: {len(second_passes):,}")
print(f"  changed the cut point:  {cut_flagged:>4} flagged, {cut_moved:>4} with a moved cut_turn")
print(f"  changed the boundaries: {bounds_flagged:>4} flagged, {bounds_moved:>4} with a moved start/end_turn")
print(f"      moved start_turn:   {start_moved:>4}")
print(f"      moved end_turn:     {end_moved:>4}")
print(f"      moved both edges:   {both_moved:>4}")

# An adjudicator sees the moment as the second pass left it, so their final call
# is compared against the boundaries after any redraw above, not the selector's.
n_adjudications = adj_cut_moved = 0
adj_start_moved = adj_end_moved = adj_both_moved = adj_bounds_moved = 0
adj_decisions = Counter()
for r in moment_rows:
    effective, _ = effective_boundaries(r["moment"], reannotation(r))
    for a in latest_annotations(r, "adjudicator"):
        payload = a["payload"]
        adj_decisions[(payload.get("decisions") or {}).get("boundaries")] += 1
        if not payload.get("final"):  # threw the moment out; drew no boundaries
            continue
        n_adjudications += 1
        adj_cut_moved += payload.get("final_cut_turn") not in (None, effective["cut_turn"])

        moved_start = payload.get("final_start_turn") not in (None, effective["start_turn"])
        moved_end = payload.get("final_end_turn") not in (None, effective["end_turn"])
        adj_start_moved += moved_start
        adj_end_moved += moved_end
        adj_both_moved += moved_start and moved_end
        adj_bounds_moved += moved_start or moved_end

print(f"\nAdjudications with a final call: {n_adjudications}")
print(f"  changed the cut point:  {adj_cut_moved:>4}")
print(f"  changed the boundaries: {adj_bounds_moved:>4}")
print(f"      moved start_turn:   {adj_start_moved:>4}")
print(f"      moved end_turn:     {adj_end_moved:>4}")
print(f"      moved both edges:   {adj_both_moved:>4}")
# `decisions.boundaries` is the adjudicator recording whose version they took:
# "agreed" (the two passes already matched), "selector" / "reannotator" (sided
# with that pass), or "combined" (drew their own -- and in 19 of 21 cases they
# did so over two passes that had already agreed). The rows below cover all
# adjudications, not just the n_adjudications above: every "not recorded" is a
# throw-out, which drew no boundaries at all.
print("\nAdjudicators' own `decisions.boundaries` (all adjudications):")
for decision, n in adj_decisions.most_common():
    print(f"  {decision or 'not recorded'}: {n}")


Second passes: 1,211
  changed the cut point:   126 flagged,  123 with a moved cut_turn
  changed the boundaries:  182 flagged,  178 with a moved start/end_turn
      moved start_turn:    115
      moved end_turn:      148
      moved both edges:     85

Adjudications with a final call: 86
  changed the cut point:     1
  changed the boundaries:   23
      moved start_turn:      4
      moved end_turn:       22
      moved both edges:      3

Adjudicators' own `decisions.boundaries` (all adjudications):
  agreed: 58
  combined: 21
  not recorded: 11
  selector: 4
  reannotator: 3


In [10]:
# Moment length after the cut point, in dialogue turns, using the boundaries as
# the second pass left them: the cut point is where generation starts and
# end_turn is where the moment stops, so `end_turn - cut_turn` is how much of
# the moment lies ahead of it.
#
# This runs over every moment anyone drew -- thrown-out and cut-point-redrawn
# ones included -- so it describes what annotators selected, not the set a tutor
# model is finally scored on. That set (the moments the combined-label section
# below keeps) is printed underneath for comparison.
after_cut = []
for r in moment_rows:
    effective, _ = effective_boundaries(r["moment"], reannotation(r))
    after_cut.append(effective["end_turn"] - effective["cut_turn"])

after_cut.sort()
median = after_cut[len(after_cut) // 2]
print(f"Dialogue turns after the cut point ({len(after_cut):,} moments)")
print(
    f"  avg: {sum(after_cut)/len(after_cut):.1f}, median: {median}, "
    f"min: {after_cut[0]}, max: {after_cut[-1]}"
)
buckets = Counter(min(n // 5 * 5, 40) for n in after_cut)
for lo in sorted(buckets):
    label = f"{lo}-{lo + 4}" if lo < 40 else "40+"
    bar = "#" * round(40 * buckets[lo] / max(buckets.values()))
    print(f"  {label:>6}: {buckets[lo]:>4}  {bar}")
print(f"  moments with nothing after the cut (end_turn == cut_turn): {after_cut.count(0)}")

# The same length over the moments that survive the throw-outs and cut-point
# redraws -- the population the label counts below are computed on.
scored = []
for r in moment_rows:
    if (
        thrown_out_by(r, "reannotator")
        or thrown_out_by(r, "adjudicator")
        or cut_point_redrawn(r)
    ):
        continue
    effective, _ = effective_boundaries(r["moment"], reannotation(r))
    scored.append(effective["end_turn"] - effective["cut_turn"])
scored.sort()
print(f"\nSame, over the {len(scored):,} moments kept for the labels below")
print(
    f"  avg: {sum(scored)/len(scored):.1f}, median: {scored[len(scored) // 2]}, "
    f"min: {scored[0]}, max: {scored[-1]}, "
    f"nothing after the cut: {scored.count(0)}"
)

# Moments per transcript, before and after the throw-outs.
selected = Counter(r["transcript_id"] for r in moment_rows)
kept_rows = [
    r
    for r in moment_rows
    if not thrown_out_by(r, "reannotator") and not thrown_out_by(r, "adjudicator")
]
kept_per_transcript = Counter(r["transcript_id"] for r in kept_rows)

print(f"\nFirst-pass moments per transcript ({len(selected)} transcripts)")
print(
    f"  avg: {sum(selected.values())/len(selected):.2f}, "
    f"min: {min(selected.values())}, max: {max(selected.values())}"
)
print(f"\nMoments nobody threw out, per transcript ({len(kept_per_transcript)} transcripts)")
print(
    f"  avg: {sum(kept_per_transcript.values())/len(kept_per_transcript):.2f}, "
    f"min: {min(kept_per_transcript.values())}, max: {max(kept_per_transcript.values())}"
)
print(f"  transcripts whose every moment was thrown out: {len(set(selected) - set(kept_per_transcript))}")

# Transcripts with no first-pass moment: the ones an annotator reviewed and
# found nothing in, plus the ones in the transcript file that carry no
# annotation record at all.
reviewed_empty = {r["transcript_id"] for r in no_key_rows}
all_transcript_ids = {t["transcript_id"] for t in transcripts}
unreviewed = all_transcript_ids - set(selected) - reviewed_empty
print(f"\nTranscripts with no moment selected: {len(all_transcript_ids) - len(selected)} / {len(all_transcript_ids)}")
print(f"  reviewed, no key moments found: {len(reviewed_empty)}")
print(f"  no annotation record at all:    {len(unreviewed)}")


Dialogue turns after the cut point (1,211 moments)
  avg: 12.2, median: 9, min: 0, max: 158
     0-4:  327  ########################################
     5-9:  310  ######################################
   10-14:  211  ##########################
   15-19:  153  ###################
   20-24:   80  ##########
   25-29:   40  #####
   30-34:   40  #####
   35-39:   19  ##
     40+:   31  ####
  moments with nothing after the cut (end_turn == cut_turn): 21

Same, over the 902 moments kept for the labels below
  avg: 12.0, median: 9, min: 0, max: 158, nothing after the cut: 15

First-pass moments per transcript (285 transcripts)
  avg: 4.25, min: 1, max: 18

Moments nobody threw out, per transcript (277 transcripts)
  avg: 3.71, min: 1, max: 15
  transcripts whose every moment was thrown out: 8

Transcripts with no moment selected: 177 / 462
  reviewed, no key moments found: 49
  no annotation record at all:    128


In [11]:
# Free-text whitespace-token lengths. The v2 payload's three prose boxes are the
# counterpart of v1's situation/action/result fields.
#
# Read through latest_annotations like everything else: a selector's re-save is
# the same person's same prose saved twice, and counting both would weight those
# annotators double.
text_fields = {
    "situation.why": ("situation", "why"),
    "action.explanation": ("action", "explanation"),
    "result.explanation": ("result", "explanation"),
}
lengths = {name: [] for name in text_fields}

for r in moment_rows:
    for a in latest_annotations(r):
        payload = answered_payload(a)
        if payload is None:  # threw the moment out; wrote none of these boxes
            continue
        for name, (section, key) in text_fields.items():
            text = (payload.get(section) or {}).get(key) or ""
            lengths[name].append(len(text.split()))

print("Free-text whitespace-token lengths (annotations that answered):")
for name, values in lengths.items():
    avg = sum(values) / len(values)
    print(f"  {name}: avg {avg:.1f}, min {min(values)}, max {max(values)}  (n={len(values):,})")

Free-text whitespace-token lengths (annotations that answered):
  situation.why: avg 30.2, min 1, max 232  (n=2,332)
  action.explanation: avg 33.9, min 1, max 240  (n=2,332)
  result.explanation: avg 30.2, min 5, max 146  (n=2,332)


In [12]:
# moment_id <-> (transcript_id, start_turn, end_turn) mapping.
# Unlike v1, every v2 moment record has a moment_id, so coverage is total; what
# is worth checking is whether the mapping to coordinates is one-to-one.
#
# It is not, and the cause is that (start_turn, end_turn) is a coarser
# coordinate than a moment. A moment's real span is start_index/end_index over
# *all* transcript rows; turn numbers only advance on dialogue rows, so any two
# moments separated by nothing but screen activity share one turn pair. Every
# collision below is a real pair of distinct selections, not a broken id:
#
#   * screen-only moments  -- the span holds no dialogue row at all, so
#     start_turn == end_turn == the last dialogue turn before it. Three such
#     moments sit between turns 71 and 72 of one transcript, two more between
#     72 and 73.
#   * a selection trimmed by screen rows -- two overlapping spans with the same
#     enclosing dialogue turns but different cut points.
#   * the same excerpt re-selected after a second pass redrew its cut point:
#     new moment_id, same span, the redrawn cut baked in.
#   * one duplicated record -- same selector, same timestamp, same span, two
#     ids, routed to two different second passes.
moment_to_coords = defaultdict(set)
coords_to_moments = defaultdict(set)

for r in moment_rows:
    m = r["moment"]
    coords = (r["transcript_id"], m.get("start_turn"), m.get("end_turn"))
    moment_to_coords[m["moment_id"]].add(coords)
    coords_to_moments[coords].add(m["moment_id"])

multi_coord = {mid: c for mid, c in moment_to_coords.items() if len(c) > 1}
multi_moment = {c: m for c, m in coords_to_moments.items() if len(m) > 1}

print(f"Unique moment_ids:                              {len(moment_to_coords)}")
print(f"Unique (transcript_id, start_turn, end_turn):   {len(coords_to_moments)}")
print(f"moment_ids mapping to >1 coord:                 {len(multi_coord)}")
print(f"coords mapping to >1 moment_id:                 {len(multi_moment)}")

# The moments a turn pair cannot describe: no dialogue row inside the span.
turns_by_transcript = {t["transcript_id"]: t["turns"] for t in transcripts}
screen_only = [
    r
    for r in moment_rows
    if not any(
        t["type"] == "dialogue"
        for t in turns_by_transcript[r["transcript_id"]][
            r["moment"]["start_index"] : r["moment"]["end_index"] + 1
        ]
    )
]
print(f"\nMoments whose span holds no dialogue row:        {len(screen_only)}")

# The colliding coordinates, with the index spans that distinguish them.
by_coords = defaultdict(list)
for r in moment_rows:
    m = r["moment"]
    by_coords[(r["transcript_id"], m.get("start_turn"), m.get("end_turn"))].append(r)

print("\nColliding coordinates:")
for coords in multi_moment:
    transcript_id, start_turn, end_turn = coords
    print(f"  {transcript_id[:8]} turns {start_turn}-{end_turn}:")
    for r in sorted(by_coords[coords], key=lambda r: r["moment"]["start_index"]):
        m = r["moment"]
        print(
            f"    {m['moment_id'][:8]}  rows {m['start_index']}-{m['end_index']},"
            f" cut turn {m['cut_turn']}, selected {m['created_at']}"
        )


Unique moment_ids:                              1211
Unique (transcript_id, start_turn, end_turn):   1205
moment_ids mapping to >1 coord:                 0
coords mapping to >1 moment_id:                 5

Moments whose span holds no dialogue row:        11

Colliding coordinates:
  5f2f6126 turns 71-71:
    3569c214  rows 99-102, cut turn 71, selected 2026-08-14T10:44:36Z
    02e89625  rows 108-116, cut turn 71, selected 2026-08-14T11:00:16Z
    318581a8  rows 121-127, cut turn 71, selected 2026-08-14T11:07:00Z
  5f2f6126 turns 72-72:
    98fcd111  rows 151-167, cut turn 72, selected 2026-08-14T11:19:19Z
    0ba6d6bf  rows 179-189, cut turn 72, selected 2026-08-14T11:25:27Z
  729bd8bc turns 223-229:
    a6730cc0  rows 268-275, cut turn 225, selected 2026-08-17T06:38:41Z
    15156888  rows 268-275, cut turn 225, selected 2026-08-17T06:38:41Z
  80cc04d9 turns 136-149:
    78742065  rows 253-269, cut turn 136, selected 2026-08-24T23:57:17Z
    3d21ee82  rows 253-269, cut turn 138, selec

## Split statistics (`tutormoments_build/v2/splits.json`)

In [13]:
manifest = json.loads(SPLITS.read_text())
assignments = manifest["assignments"]
print(f"Transcripts assigned: {len(assignments)}")

by_split = defaultdict(list)
for r in moment_rows:
    split = assignments.get(r["transcript_id"], {}).get("split")
    by_split[split].append(r)

for split in ("iterate", "heldout", None):
    records = by_split.get(split, [])
    if not records:
        continue
    label = split or "unassigned"
    n_transcripts = len({r["transcript_id"] for r in records})
    print(f"{label:<12} {n_transcripts:>4} transcripts, {len(records):>5} moments")

Transcripts assigned: 329
iterate       137 transcripts,   590 moments
heldout       145 transcripts,   597 moments
unassigned      3 transcripts,    24 moments


In [14]:
# Annotator coverage per moment. Counted over latest_annotations, so staff and
# re-saves are out on the same terms as everywhere else; taking a set of ids off
# the raw list gives the same numbers here only because this export has no staff
# annotations and nobody holds two roles on one moment.
def distinct_annotators(record):
    return {a["annotator_id"] for a in latest_annotations(record)}


for split in ("iterate", "heldout"):
    records = by_split.get(split, [])
    if not records:
        continue
    per_moment = [len(distinct_annotators(r)) for r in records]
    coverage = Counter(per_moment)
    avg = sum(per_moment) / len(per_moment)
    print(f"{split}: {len(records)} moments, {avg:.2f} annotators/moment on avg")
    for n in sorted(coverage):
        print(f"    {n} annotator(s): {coverage[n]:>5} ({100*coverage[n]/len(records):.1f}%)")

iterate: 590 moments, 2.10 annotators/moment on avg
    2 annotator(s):   550 (93.2%)
    3 annotator(s):    23 (3.9%)
    4 annotator(s):    17 (2.9%)
heldout: 597 moments, 2.07 annotators/moment on avg
    2 annotator(s):   565 (94.6%)
    3 annotator(s):    24 (4.0%)
    4 annotator(s):     8 (1.3%)


## Combined labels over multiply annotated moments

Each moment is annotated by two or more people. This section collapses them into
one label set per moment and counts the moments that come out.

Filter out: 
- moments any annotator flagged as throw out
- moment where cut points are redrawn 

`EXCLUDED_ANNOTATORS` should be dropped. Take the highest revision per (annotator, role). 
Adjudicators' annotations are in `payload["final"]`, and each counts as
one more vote. 

**Types of situation**. Report counts for: 
- all annotators say scaffolding is appropriate (scaffold_only)
- majority of annotators say scaffolding is appropriate (scaffold_maj)
- 50/50 split of scaffolding and rigor is appropriate (fifty_fifty)
- majority of annotators say rigor is appropriate (rigor_maj)
- all annotators say rigor is appropriate (rigor_only)

**Actions are combined by majority vote.**
Most moments have exactly two annotators, so many votes tie. Tie→True is the union
(anyone saying yes makes it yes); tie→False is the intersection (everyone must
say yes). Report counts for action direction (scaffolding or a push for rigor is present.)

Over-scaffolding is the annotator's literal `scaffolding_amount ==
"over_scaffolding"` choice, which is the only over-scaffolding question anyone
answers. Here, report counts for: 
- at least one annotator says over-scaffolding is present
- majority says over-scaffolding is present

In [15]:
from tutormoments_build.v2.adjudicator_agreement import payload_labels

INCLUDE_THROWN_OUT = False
INCLUDE_CUT_POINT_REDRAWN = False

SITUATION_FIELDS = ("scaffolding_appropriate", "rigor_appropriate")


def moment_votes(record):
    """Every usable vote on one moment.

    Staff are dropped and re-saves collapsed by latest_annotations; anyone who
    threw the moment out answered nothing and abstains rather than voting no.
    """
    payloads = (answered_payload(a) for a in latest_annotations(record))
    return [payload_labels(p) for p in payloads if p is not None]


kept = moment_rows
n_thrown = n_redrawn = 0

if not INCLUDE_THROWN_OUT:
    n_thrown = sum(
        1
        for r in kept
        if thrown_out_by(r, "reannotator") or thrown_out_by(r, "adjudicator")
    )
    kept = [
        r
        for r in kept
        if not thrown_out_by(r, "reannotator") and not thrown_out_by(r, "adjudicator")
    ]

if not INCLUDE_CUT_POINT_REDRAWN:
    n_redrawn = sum(1 for r in kept if cut_point_redrawn(r))
    kept = [r for r in kept if not cut_point_redrawn(r)]

voted = [(r, v) for r, v in ((r, moment_votes(r)) for r in kept) if v]
total = len(voted)

print(f"Moment records:                  {len(moment_rows)}")
print(f"  thrown out, dropped:           {n_thrown}")
print(f"  cut point redrawn, dropped:    {n_redrawn}")
print(f"  no usable vote, dropped:       {len(kept) - total}")
print(f"Moments combined below:          {total}")
print(f"  with 1 annotators:   {sum(1 for _, v in voted if len(v) == 1)}")
print(f"  with 2 annotators:  {sum(1 for _, v in voted if len(v) == 2)}")
print(f"  with 3+ annotators: {sum(1 for _, v in voted if len(v) >= 3)}")


Moment records:                  1211
  thrown out, dropped:           183
  cut point redrawn, dropped:    126
  no usable vote, dropped:       0
Moments combined below:          902
  with 1 annotators:   0
  with 2 annotators:  847
  with 3+ annotators: 55


In [16]:
# Types of situation. Each annotator answers scaffolding_appropriate and
# rigor_appropriate independently, so one annotator can call for both; nobody in
# this export calls for neither. Counting votes per side puts every moment in
# exactly one of five rows: unanimous for one side with nobody for the other,
# a majority for one side, or an even split.
def situation_bucket(votes):
    scaffold = sum(1 for v in votes if v["scaffolding_appropriate"])
    rigor = sum(1 for v in votes if v["rigor_appropriate"])
    if scaffold == len(votes) and rigor == 0:
        return "scaffold_only"
    if rigor == len(votes) and scaffold == 0:
        return "rigor_only"
    if scaffold > rigor:
        return "scaffold_maj"
    if rigor > scaffold:
        return "rigor_maj"
    return "fifty_fifty"


BUCKET_LABELS = {
    "scaffold_only": "all say scaffolding, none rigor",
    "scaffold_maj": "majority say scaffolding",
    "fifty_fifty": "even split",
    "rigor_maj": "majority say rigor",
    "rigor_only": "all say rigor, none scaffolding",
}

buckets = Counter(situation_bucket(v) for _, v in voted)
print(f"Situation, votes per side ({total} moments)\n")
for name, label in BUCKET_LABELS.items():
    n = buckets[name]
    print(f"  {name:<14} {label:<34} {n:>5} ({100 * n / total:.1f}%)")
print(f"  {'-' * 60}")
print(f"  {'total':<14} {'':<34} {sum(buckets.values()):>5}")

# How often a single annotator called for both, which is what puts a moment in
# an even split without two annotators actually disagreeing.
all_votes = [v for _, votes in voted for v in votes]
both = sum(1 for v in all_votes if v["scaffolding_appropriate"] and v["rigor_appropriate"])
neither = sum(
    1 for v in all_votes if not v["scaffolding_appropriate"] and not v["rigor_appropriate"]
)
print(f"\nAnnotator votes: {len(all_votes)} — {both} called for both, {neither} for neither")


Situation, votes per side (902 moments)

  scaffold_only  all say scaffolding, none rigor      293 (32.5%)
  scaffold_maj   majority say scaffolding              53 (5.9%)
  fifty_fifty    even split                           263 (29.2%)
  rigor_maj      majority say rigor                    40 (4.4%)
  rigor_only     all say rigor, none scaffolding      253 (28.0%)
  ------------------------------------------------------------
  total                                               902

Annotator votes: 1877 — 76 called for both, 0 for neither


In [17]:
# Action: majority vote, under both tie rules. Most moments have exactly two
# annotators, so most votes tie; the tie rule is not a detail, it is most of the
# answer. Tie->True is the union (anyone saying yes makes it yes); tie->False is
# the intersection (everyone must say yes).
def majority(values, tie):
    yes = sum(1 for v in values if v)
    no = len(values) - yes
    if yes == no:
        return tie
    return yes > no


ACTION_LABELS = {
    "scaffolding_present": "scaffolded (any level)",
    "rigor_present": "pushed for rigor",
}

print(f"Action direction, majority vote ({total} moments)\n")
print(f"  {'':<28}{'tie→True':>14}{'tie→False':>14}{'ties':>10}")
print(f"  {'-' * 66}")
for field, label in ACTION_LABELS.items():
    n_true = sum(1 for _, votes in voted if majority([v[field] for v in votes], True))
    n_false = sum(1 for _, votes in voted if majority([v[field] for v in votes], False))
    print(
        f"  {label:<28}{n_true:>6} ({100 * n_true / total:>4.1f}%)"
        f"{n_false:>6} ({100 * n_false / total:>4.1f}%){n_true - n_false:>10}"
    )

# Over-scaffolding is the annotator's literal scaffolding_amount ==
# "over_scaffolding" choice, the only over-scaffolding question anyone answers.
# build_ground_truth additionally *infers* over-scaffolding from resolved labels
# (scaffolding present where scaffolding was not appropriate); that case is
# derived rather than voted on, so these numbers are lower than the ground truth's.
field = "over_scaffolding_declared"
at_least_one = sum(1 for _, votes in voted if any(v[field] for v in votes))
majority_yes = sum(1 for _, votes in voted if majority([v[field] for v in votes], False))
print(f"\nOver-scaffolding, declared ({total} moments)\n")
print(f"  at least one annotator: {at_least_one:>5} ({100 * at_least_one / total:.1f}%)")
print(f"  majority (tie→False):   {majority_yes:>5} ({100 * majority_yes / total:.1f}%)")


Action direction, majority vote (902 moments)

                                    tie→True     tie→False      ties
  ------------------------------------------------------------------
  scaffolded (any level)         686 (76.1%)   454 (50.3%)       232
  pushed for rigor               417 (46.2%)   178 (19.7%)       239

Over-scaffolding, declared (902 moments)

  at least one annotator:   331 (36.7%)
  majority (tie→False):      88 (9.8%)
